# `llm-data` pipeline walkthrough

This notebook connects the repository's core components without downloading data or models. The default path uses a tiny synthetic Daft DataFrame, runs on CPU, and is deterministic. It demonstrates interfaces and stage boundaries, not production quality thresholds.

## Architecture and workflow map

| Order | Stage | Repository surface | Operation kind | Default demo |
|---|---|---|---|---|
| 1 | Acquisition/loading | `DataLoader`; Daft readers | I/O and checkpoint-aware loading | Synthetic rows replace remote WARC input |
| 2 | WARC extraction | `ExtractWarc` | Row-level schema transform | Shown in an optional production cell |
| 3 | HTML parsing | `ParseHtml` | Row-level transform | Executed with local HTML |
| 4 | Encoding normalization | `FixEncoding` | Row-level transform | Executed with a mojibake example |
| 5 | Cheap filtering | `URLBlacklistFilter`, `LengthFilter`, `C4PageFilter` | Row-level filtering/policy | Executed with an in-memory blacklist |
| 6 | Optional enrichment | `ExtractLanguage`, `PIIAnonymizer`, `EmbedText` | Model/resource-backed transforms | Fenced behind `RUN_OPTIONAL_MODELS` |
| 7 | AI-content review | `AIContentScorer` then `AIDomainReviewReport` | Model inference then corpus aggregation/review artifact | Executed with an injected fake predictor |
| 8 | Human-approved policy | `ReviewedAIDomainFilter` | Final filtering policy | Executed with a synthetic reviewed-domain set |
| 9 | Deduplication | `dedupe_3_sentences`; `fuzzy_dedupe` | Corpus-level operation | Small C4-style dedupe executes; fuzzy dedupe is optional |
| 10 | Output/checkpointing | Daft Parquet writes; loader checkpoints | Materialization and I/O | Shown in an optional production cell |

`DataEngine` composes callables that preserve a compatible page-row schema. Aggregations, model-backed stages, path-oriented distributed deduplication, and writes are clearer as explicit boundaries with separately named DataFrames.

In [ ]:
RUN_OPTIONAL_MODELS = False

import math
from pprint import pprint

import daft

from llm_data.encoding import FixEncoding
from llm_data.engine import DataEngine
from llm_data.factories.ai_content import AIContentScorer
from llm_data.filters.ai_content import ReviewedAIDomainFilter
from llm_data.filters.c4.dedupe import dedupe_3_sentences
from llm_data.filters.c4.page_rules import C4PageFilter
from llm_data.filters.filters import LengthFilter
from llm_data.filters.url_blacklist import URLBlacklistFilter
from llm_data.parsers.html import ParseHtml
from llm_data.reports.ai_content import AIDomainReviewReport

## 1. A realistic, local input boundary

A production WARC reader yields WARC headers and bytes before `ExtractWarc` creates `html`, `url`, `record_id`, and provenance columns. The synthetic input starts immediately after that boundary so it needs no network or Common Crawl files. It includes duplicate pages, broken encoding, a blacklisted URL, lorem ipsum, a short page, and a domain that will become an AI-content review candidate.

In [ ]:
def article_html(text: str) -> str:
    return f"<html><head><title>Example</title></head><body><article><p>{text}</p></article></body></html>"


duplicate_text = (
    "A practical data pipeline validates each boundary before scaling. "
    "It preserves source identifiers and makes filtering decisions auditable. "
    "The final caf\u00c3\u00a9 example also exercises encoding repair."
)
generated_marker_text = (
    "This synthetic-marker page is only a deterministic scoring fixture. "
    "synthetic-marker appears again so the fake predictor emits a high score. "
    "The score is not evidence or ground truth about real authorship."
)
ordinary_domain_text = (
    "This ordinary article shares the candidate domain but has no scoring marker. "
    "It demonstrates why domain reports require thresholds and human review."
)

raw_pages = daft.from_pylist([
    {"record_id": "a1", "url": "https://docs.example/one", "html": article_html(duplicate_text), "source_path": "synthetic.warc"},
    {"record_id": "a2", "url": "https://docs.example/two", "html": article_html(duplicate_text), "source_path": "synthetic.warc"},
    {"record_id": "g1", "url": "https://generated.example/one", "html": article_html(generated_marker_text), "source_path": "synthetic.warc"},
    {"record_id": "g2", "url": "https://generated.example/two", "html": article_html(generated_marker_text + " Another synthetic-marker."), "source_path": "synthetic.warc"},
    {"record_id": "g3", "url": "https://generated.example/three", "html": article_html(ordinary_domain_text), "source_path": "synthetic.warc"},
    {"record_id": "b1", "url": "https://ads.blocked.example/page", "html": article_html(duplicate_text), "source_path": "synthetic.warc"},
    {"record_id": "l1", "url": "https://docs.example/lorem", "html": article_html("Lorem ipsum " + duplicate_text), "source_path": "synthetic.warc"},
    {"record_id": "s1", "url": "https://docs.example/short", "html": article_html("Too short."), "source_path": "synthetic.warc"},
])

assert raw_pages.count_rows() == 8

## 2. Compose compatible row-level transforms

These components all accept and return a page-shaped Daft DataFrame, so they fit a linear `DataEngine`. The blacklist is passed in memory; omitting it would invoke the repository's Hugging Face dataset download. The C4 component is configured to exercise its lorem-ipsum page rule without imposing production sentence-line thresholds on tiny HTML fixtures.

`collect()` deliberately materializes a snapshot after cheap parsing and filtering. In a production job this is a useful place to inspect the schema, record metrics, or write a resumable checkpoint.

In [ ]:
row_engine = DataEngine(
    components=[
        ParseHtml(parser_type="trafilatura"),
        FixEncoding(),
        URLBlacklistFilter(blacklist={"blocked.example"}),
        LengthFilter(min_len=80, max_len=2_000),
        C4PageFilter(
            lorem_ipsum_filter=True,
            line_terminal_punctuation_filter=False,
            line_min_word_filter=False,
            page_min_line_filter=False,
        ),
    ],
    name="local_row_preparation",
)

prepared_pages = row_engine.run(raw_pages).collect()
prepared = prepared_pages.to_pydict()
assert sorted(prepared["record_id"]) == ["a1", "a2", "g1", "g2", "g3"]
assert all("cafe" not in text or "caf\u00e9" in text for text in prepared["text"])
pprint(sorted(zip(prepared["record_id"], prepared["url"])))

## 3. Keep model inference separate from filtering policy

`AIContentScorer` supports an injected predictor factory. The fake predictor below implements the same label/logit protocol as a sequence classifier but uses only a marker count, so no model, GPU, or network is involved. Scores are model-specific signals, never ground truth.

In [ ]:
class FakePredictor:
    @property
    def id2label(self):
        return {0: "human", 1: "generated"}

    def __call__(self, texts):
        logits = []
        for text in texts:
            marker_count = text.lower().count("synthetic-marker")
            logits.append([0.0, 2.0 * marker_count - 2.0])
        return logits


scoring_engine = DataEngine(
    components=[
        AIContentScorer(
            model_name_or_path="unused-with-injected-predictor",
            ai_label="generated",
            batch_size=4,
            gpus=0,
            predictor_factory=FakePredictor,
        )
    ],
    name="offline_model_scoring",
)
scored_pages = scoring_engine.run(prepared_pages).collect()
scores = scored_pages.to_pydict()
score_by_id = dict(zip(scores["record_id"], scores["ai_content_score"]))
assert score_by_id["g1"] > 0.8 and score_by_id["g2"] > 0.8
assert score_by_id["g3"] < 0.5
pprint({key: round(value, 4) for key, value in sorted(score_by_id.items())})

## 4. Score -> candidate-domain report -> human review -> policy

`AIDomainReviewReport` is a corpus aggregation: it changes the schema from pages to domain statistics. It therefore should not be inserted into a page-row `DataEngine`. Materialize or publish this report for reviewers, then feed only the approved domain set back into `ReviewedAIDomainFilter` on the scored page snapshot.

The synthetic `reviewed_domains` assignment stands in for an external human decision. A candidate report alone must never automatically remove a domain.

In [ ]:
candidate_report = AIDomainReviewReport(
    page_score_threshold=0.8,
    min_pages=3,
    min_candidate_fraction=2 / 3,
)(scored_pages).collect()

report = candidate_report.to_pydict()
assert report["domain"] == ["generated.example"]
assert report["page_count"] == [3]
assert report["pages_above_threshold"] == [2]
assert math.isclose(report["fraction_above_threshold"][0], 2 / 3)
pprint(report)

In [ ]:
# In production this set comes from a versioned review artifact, not from scores alone.
reviewed_domains = {"generated.example"}
policy_filtered_pages = ReviewedAIDomainFilter(reviewed_domains)(scored_pages).collect()
remaining = policy_filtered_pages.to_pydict()
assert sorted(remaining["record_id"]) == ["a1", "a2"]

## 5. Cross-row deduplication and final projection

`dedupe_3_sentences` is a corpus-level operation rather than an independent row transform. It explodes text into spans, compares rows with a window, and joins the novel document IDs back to the page schema. The two remaining synthetic pages have identical text, so the lower `record_id` is retained.

In [ ]:
deduplicated_pages = dedupe_3_sentences(policy_filtered_pages).collect()
final_pages = deduplicated_pages.select(
    "record_id", "url", "text", "source_path", "ai_content_score"
).collect()
final = final_pages.to_pydict()
assert final["record_id"] == ["a1"]
pprint(final)

## 6. Optional production WARC loading and checkpointing

This cell documents the real acquisition boundary. It remains inert by default because the paths, WARC objects, and checkpoint store are deployment concerns. `ExtractWarc` expects Daft's WARC columns and emits the page schema used above.

In [ ]:
if RUN_OPTIONAL_MODELS:
    from llm_data.loader import DataLoader
    from llm_data.parsers.warc import ExtractWarc

    warc_records = DataLoader(
        loader_type="warc",
        checkpoint_path="s3://bucket/checkpoints/warc-read",
        checkpoint_on="source_path",
    ).read_data("s3://bucket/common-crawl/*.warc.gz")
    extracted_pages = DataEngine(
        components=[ExtractWarc()], name="warc_extraction"
    ).run(warc_records)
    extracted_pages.write_parquet(
        "s3://bucket/checkpoints/extracted-pages",
        write_mode="overwrite",
    )

## 7. Optional model-backed enrichment

These components are intentionally outside the default path:

- `AIContentScorer` downloads a configured Hugging Face classifier unless a predictor is injected. Validate its `id2label`, normalization, and thresholds on representative data.
- `ExtractLanguage` downloads GlotLID during construction and uses fastText inference.
- `PIIAnonymizer` constructs Presidio analyzer/anonymizer resources. Decide whether masking or filtering is the dataset policy before applying it.
- `EmbedText` currently asserts CUDA availability and configures a GPU-oriented SentenceTransformer path.

All are gated so **Run All** remains offline and CPU-safe.

In [ ]:
if RUN_OPTIONAL_MODELS:
    from llm_data.factories.embedding import EmbedText
    from llm_data.filters.pii import PIIAnonymizer
    from llm_data.language_id.lang_id import ExtractLanguage

    real_scored_pages = AIContentScorer(
        model_name_or_path="organization/validated-ai-content-detector",
        ai_label="generated",
        normalization="softmax",
        batch_size=32,
        gpus=1,
    )(prepared_pages)
    language_tagged_pages = ExtractLanguage()(prepared_pages)
    pii_masked_pages = PIIAnonymizer(language="en")(prepared_pages)
    embedded_pages = EmbedText(
        model_name="lightonai/DenseOn",
        embedding_dim=128,
        gpus=1,
    )(prepared_pages)

## 8. Optional fuzzy and embedding deduplication

`fuzzy_dedupe` is a path-oriented distributed batch job: it forces the Ray runner, writes signatures/edges/removal sets, and atomically publishes directories. That lifecycle is incompatible with an in-memory row pipeline, so checkpoint pages first and invoke it as a separate job. Issue [#19](https://github.com/teraflop-ai/llm-data/issues/19) tracks embedding deduplication; this repository does not yet expose a corresponding end-to-end embedding-dedup component.

In [ ]:
if RUN_OPTIONAL_MODELS:
    from llm_data.deduplication.deduplicate import fuzzy_dedupe

    # A prior job writes prepared pages with doc_id and paragraph_text columns.
    fuzzy_dedupe(
        input_path="s3://bucket/checkpoints/pages",
        work="s3://bucket/work/fuzzy-dedupe",
        output_dir="s3://bucket/output/fuzzy-deduplicated",
        key="doc_id",
        text="paragraph_text",
    )

## 9. Production ordering and contributor opportunities

A practical production plan is:

1. Load source objects with provenance and checkpoint keys.
2. Extract WARC records, parse HTML, normalize encoding, and materialize a page-schema checkpoint.
3. Apply cheap URL, length, and heuristic filters; record per-rule counts.
4. Run exact/C4-style deduplication where its effect on later domain statistics is understood.
5. Run optional PII/language/model/embedding enrichment as separately resourced jobs.
6. Publish AI-score domain aggregates as candidate **review artifacts**, collect human decisions, and apply only the versioned reviewed blocklist as policy.
7. Run path-oriented fuzzy or future embedding deduplication, then write versioned outputs and manifests.

Current boundaries and open work are visible rather than hidden: `DataEngine` is a linear callable composer with manually managed schemas/materialization; the production WARC, model, report-review, distributed dedup, and output lifecycles remain separate. Open issues cover broader heuristic filtering ([#8](https://github.com/teraflop-ai/llm-data/issues/8), [#15](https://github.com/teraflop-ai/llm-data/issues/15), [#25](https://github.com/teraflop-ai/llm-data/issues/25), [#26](https://github.com/teraflop-ai/llm-data/issues/26), [#34](https://github.com/teraflop-ai/llm-data/issues/34)), embedding deduplication ([#19](https://github.com/teraflop-ai/llm-data/issues/19)), SEO/quality/model scoring ([#22](https://github.com/teraflop-ai/llm-data/issues/22), [#27](https://github.com/teraflop-ai/llm-data/issues/27), [#30](https://github.com/teraflop-ai/llm-data/issues/30), [#35](https://github.com/teraflop-ai/llm-data/issues/35)), and specialized extraction ([#29](https://github.com/teraflop-ai/llm-data/issues/29), [#31](https://github.com/teraflop-ai/llm-data/issues/31)).

Model-backed signals should remain separate from filtering policy, and review reports should remain versioned artifacts rather than implicit side effects.